# 01 — Schema exploration

A first pass at the MIMIC-IV Demo: what's in it, how it's organized, and what we can pull out of it. I use DuckDB to query the gzipped CSVs directly — lightweight on the demo (~100 patients), and the same code will scale to full MIMIC-IV once credentialed.

## Setup

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

DATA_DIR = Path("../data")
HOSP = DATA_DIR / "hosp"
ICU = DATA_DIR / "icu"

con = duckdb.connect()

## Available tables

In [2]:
hosp_files = sorted([f.name for f in HOSP.glob("*.csv.gz")])
icu_files = sorted([f.name for f in ICU.glob("*.csv.gz")])

print("hosp module:")
for f in hosp_files:
    print(f"  {f}")

print("\nicu module:")
for f in icu_files:
    print(f"  {f}")

hosp module:
  admissions.csv.gz
  d_hcpcs.csv.gz
  d_icd_diagnoses.csv.gz
  d_icd_procedures.csv.gz
  d_labitems.csv.gz
  diagnoses_icd.csv.gz
  drgcodes.csv.gz
  emar.csv.gz
  emar_detail.csv.gz
  hcpcsevents.csv.gz
  labevents.csv.gz
  microbiologyevents.csv.gz
  omr.csv.gz
  patients.csv.gz
  pharmacy.csv.gz
  poe.csv.gz
  poe_detail.csv.gz
  prescriptions.csv.gz
  procedures_icd.csv.gz
  provider.csv.gz
  services.csv.gz
  transfers.csv.gz

icu module:
  caregiver.csv.gz
  chartevents.csv.gz
  d_items.csv.gz
  datetimeevents.csv.gz
  icustays.csv.gz
  ingredientevents.csv.gz
  inputevents.csv.gz
  outputevents.csv.gz
  procedureevents.csv.gz


MIMIC-IV is split into two modules:

- `hosp/` — hospital-wide tables (patient demographics, admissions, diagnoses, prescriptions, labs).
- `icu/` — ICU-specific tables (ICU stays, charted vitals, inputs/outputs).

For this project I care about: `patients` (demographics), `admissions` (visit-level info incl. outcome), `icustays` (ICU stay boundaries), `chartevents` (vitals over time), `labevents` (labs).

## Patients

In [3]:
patients = con.execute(f"SELECT * FROM '{HOSP / 'patients.csv.gz'}'").df()
print(f"shape: {patients.shape}")
patients.head()

shape: (100, 6)


,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod
0,10014729,F,21,2125,2011 - 2013,NaT
1,10003400,F,72,2134,2011 - 2013,2137-09-02
2,10002428,F,80,2155,2011 - 2013,NaT
3,10032725,F,38,2143,2011 - 2013,2143-03-30
4,10027445,F,48,2142,2011 - 2013,2146-02-09


In [4]:
patients.dtypes

subject_id                    int64
gender                          str
anchor_age                    int64
anchor_year                   int64
anchor_year_group               str
dod                  datetime64[us]
dtype: object

In [5]:
patients.isna().mean().sort_values(ascending=False).head(10)

dod                  0.69
subject_id           0.00
gender               0.00
anchor_age           0.00
anchor_year          0.00
anchor_year_group    0.00
dtype: float64

Each row = one patient. Key columns: `subject_id`, `gender`, `anchor_age`, `anchor_year`. MIMIC-IV anchors age for privacy reasons — `anchor_age` is what I'll use to filter adults later.

## Admissions

In [6]:
admissions = con.execute(f"SELECT * FROM '{HOSP / 'admissions.csv.gz'}'").df()
print(f"shape: {admissions.shape}")
admissions.head()

shape: (275, 16)


,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag
0,10004235,24181354,2196-02-24 14:38:00,2196-03-04 14:02:00,NaT,URGENT,P03YMR,TRANSFER FROM HOSPITAL,SKILLED NURSING FACILITY,Medicaid,ENGLISH,SINGLE,BLACK/CAPE VERDEAN,2196-02-24 12:15:00,2196-02-24 17:07:00,0
1,10009628,25926192,2153-09-17 17:08:00,2153-09-25 13:20:00,NaT,URGENT,P41R5N,TRANSFER FROM HOSPITAL,HOME HEALTH CARE,Medicaid,?,MARRIED,HISPANIC/LATINO - PUERTO RICAN,NaT,NaT,0
2,10018081,23983182,2134-08-18 02:02:00,2134-08-23 19:35:00,NaT,URGENT,P233F6,TRANSFER FROM HOSPITAL,SKILLED NURSING FACILITY,Medicare,ENGLISH,MARRIED,WHITE,2134-08-17 16:24:00,2134-08-18 03:15:00,0
3,10006053,22942076,2111-11-13 23:39:00,2111-11-15 17:20:00,2111-11-15 17:20:00,URGENT,P38TI6,TRANSFER FROM HOSPITAL,DIED,Medicaid,ENGLISH,NaN,UNKNOWN,NaT,NaT,1
4,10031404,21606243,2113-08-04 18:46:00,2113-08-06 20:57:00,NaT,URGENT,P07HDB,TRANSFER FROM HOSPITAL,HOME,Other,ENGLISH,WIDOWED,WHITE,NaT,NaT,0


In [7]:
# In-hospital mortality is captured by hospital_expire_flag
admissions["hospital_expire_flag"].value_counts()

hospital_expire_flag
0    260
1     15
Name: count, dtype: int64

`hospital_expire_flag` is my outcome: 1 if the patient died in hospital, 0 otherwise. Other useful columns: `admittime`, `dischtime`, `deathtime`, `race`, `insurance`, `marital_status`. The demographic columns are what I'll use to define subgroups in the fairness audit.

## ICU stays

In [8]:
icustays = con.execute(f"SELECT * FROM '{ICU / 'icustays.csv.gz'}'").df()
print(f"shape: {icustays.shape}")
icustays.head()

shape: (140, 8)


,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los
0,10018328,23786647,31269608,Neuro Stepdown,Neuro Stepdown,2154-04-24 23:03:44,2154-05-02 15:55:21,7.702512
1,10020187,24104168,37509585,Neuro Surgical Intensive Care Unit (Neuro SICU),Neuro Stepdown,2169-01-15 04:56:00,2169-01-20 15:47:50,5.452662
2,10020187,26842957,32554129,Neuro Intermediate,Neuro Intermediate,2170-02-24 18:18:46,2170-02-25 15:15:26,0.872685
3,10012853,27882036,31338022,Trauma SICU (TSICU),Trauma SICU (TSICU),2176-11-26 02:34:49,2176-11-29 20:58:54,3.766725
4,10020740,25826145,32145159,Trauma SICU (TSICU),Trauma SICU (TSICU),2150-06-03 20:12:32,2150-06-04 21:05:58,1.037106


In [9]:
icustays[["los", "first_careunit"]].describe(include="all")

,los,first_careunit
count,140.000000,140
unique,NaN,9
top,NaN,Medical Intensive Care Unit (MICU)
freq,NaN,29
mean,3.679379,NaN
std,3.896354,NaN
min,0.023727,NaN
25%,1.170663,NaN
50%,2.155093,NaN
75%,4.907749,NaN


`icustays` gives me ICU stay boundaries. `intime` and `outtime` are what I'll use to extract the "first 24h" features in notebook 04. `los` (length of stay in days) is also worth keeping in mind.

## Quick peek at the time-series tables

In [10]:
chartevents_preview = con.execute(
    f"SELECT * FROM '{ICU / 'chartevents.csv.gz'}' LIMIT 5"
).df()
chartevents_preview

,subject_id,hadm_id,stay_id,caregiver_id,charttime,storetime,itemid,value,valuenum,valueuom,warning
0,10005817,20626031,32604416,6770,2132-12-16,2132-12-15 23:45:00,225054,On,NaN,NaN,0
1,10005817,20626031,32604416,6770,2132-12-16,2132-12-15 23:43:00,223769,100,100.0,%,0
2,10005817,20626031,32604416,6770,2132-12-16,2132-12-15 23:47:00,223956,Atrial demand,NaN,NaN,0
3,10005817,20626031,32604416,6770,2132-12-16,2132-12-15 23:47:00,224866,Yes,NaN,NaN,0
4,10005817,20626031,32604416,6770,2132-12-16,2132-12-15 23:45:00,227341,No,0.0,NaN,0


In [11]:
labevents_preview = con.execute(
    f"SELECT * FROM '{HOSP / 'labevents.csv.gz'}' LIMIT 5"
).df()
labevents_preview

,labevent_id,subject_id,hadm_id,specimen_id,itemid,order_provider_id,charttime,storetime,value,valuenum,valueuom,ref_range_lower,ref_range_upper,flag,priority,comments
0,172061,10014354,29600294,1808066,51277,None,2148-08-16,2148-08-16 01:30:00,15.4,15.40,%,10.5,15.5,NaN,ROUTINE,None
1,172062,10014354,29600294,1808066,51279,None,2148-08-16,2148-08-16 01:30:00,3.35,3.35,m/uL,4.6,6.1,abnormal,ROUTINE,None
2,172068,10014354,29600294,1808066,52172,None,2148-08-16,2148-08-16 01:30:00,49.7,49.70,fL,35.1,46.3,abnormal,ROUTINE,None
3,172063,10014354,29600294,1808066,51301,None,2148-08-16,2148-08-16 01:30:00,20.3,20.30,K/uL,4.0,10.0,abnormal,ROUTINE,None
4,172050,10014354,29600294,1808066,51249,None,2148-08-16,2148-08-16 01:30:00,31.1,31.10,g/dL,32.0,37.0,abnormal,ROUTINE,None


Both are long-format: one row per (patient, time, measurement). To build features for a 24h window I'll join on `subject_id` + `stay_id` and filter on time relative to `intime` — that's notebook 04.

## Takeaways

- Demo has ~100 patients across `hosp` and `icu`.
- Outcome (`hospital_expire_flag`) lives in `admissions`.
- Demographics are split: `patients` holds `gender` and `anchor_age`; `admissions` holds `race`, `insurance`, `marital_status`.
- Time-series tables are long-format — feature extraction will need a windowed join on `intime`.

Next (notebook 02): demographic distributions, outcome rates, and a first look at how outcomes break down by subgroup.